# RetinaSafe — Step 6: DINOv2 linear probe on BRSET

**Goal:** evaluate **frozen DINOv2 ViT-L/14** (general-purpose vision foundation model, trained on ~142M natural images via self-supervised learning) as a feature extractor for BRSET 12-label multilabel classification. Compare head-to-head against the RETFound linear probe from Step 5.

**The research question this answers:** does domain-specific pretraining (RETFound on 1.6M retinal images) actually beat general-purpose pretraining (DINOv2 on 142M natural images) for fundus multilabel?

**Three possible outcomes — all interesting:**
- DINOv2 > RETFound → domain-specific pretraining is overrated; just use bigger general models
- DINOv2 < RETFound → domain knowledge matters; the probe gap is a probe-issue, not a RETFound-issue
- DINOv2 ≈ RETFound → neither is enough; fine-tuning is what carries weight

**Inputs:**
1. `samarthmishra208/brset-brazilian-multilabel-ophthalmological`
2. **Notebook Output:** `samarthmishra208/brset-baseline` (for `brset_index.csv`)

**Settings:** GPU T4 x2, **Internet ON** (first run downloads DINOv2 from torch.hub, ~1.1 GB). Runtime ~25–30 min.

**Important: DINOv2 uses 14-px patches**, so image size must be a multiple of 14. We use 224×224 (16×16 = 256 patches + 1 CLS = 257 tokens).

## Cell 1 — Imports + paths

In [ ]:
import os, sys, json, time, glob, pathlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Find BRSET images + brset_index.csv (same as RETFound notebook)
img_dirs = glob.glob("/kaggle/input/**/fundus_photos", recursive=True)
assert img_dirs, "BRSET fundus_photos not found"
IMAGES_DIR = img_dirs[0]
idx_csvs = glob.glob("/kaggle/input/**/brset_index.csv", recursive=True)
assert idx_csvs, "brset_index.csv not found — attach brset-baseline notebook output"
INDEX_CSV = idx_csvs[0]
print(f"\nIMAGES_DIR: {IMAGES_DIR}")
print(f"INDEX_CSV: {INDEX_CSV}")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
FEATURES_PATH = WORK / "dinov2_features.npy"

DISEASE_COLS = [
    "diabetic_retinopathy", "macular_edema", "scar", "nevus", "amd",
    "vascular_occlusion", "hypertensive_retinopathy", "drusens",
    "hemorrhage", "myopic_fundus", "increased_cup_disc", "other",
]
N_LABELS = len(DISEASE_COLS)

## Cell 2 — Load frozen DINOv2 ViT-L/14

torch.hub downloads weights from facebookresearch/dinov2. ~1.1 GB. First run takes a couple minutes; subsequent runs use the cached copy.

In [ ]:
# torch.hub.load downloads and instantiates the model
print("Loading DINOv2 ViT-L/14...")
backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14", verbose=False)
print(f"DINOv2 loaded: {sum(p.numel() for p in backbone.parameters())/1e6:.1f}M params")

# Freeze
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE).eval()
print(f"Frozen DINOv2 on {DEVICE}")

# Sanity check output shape with a dummy batch
dummy = torch.zeros(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    feat = backbone(dummy)
print(f"Output feature shape on dummy batch: {feat.shape}  (expected: [2, 1024])")
EMBED_DIM = feat.shape[1]
print(f"Embedding dim: {EMBED_DIM}")

## Cell 3 — Dataset + transforms (DINOv2 normalization)

DINOv2 was trained with ImageNet normalization (same as RETFound — convenient). 224×224 is a multiple of 14, so it works.

In [ ]:
IMG_SIZE = 224  # multiple of 14
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

feat_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BRSETAllSplitsDataset(Dataset):
    def __init__(self, index_csv, images_dir, transform=None):
        self.df = pd.read_csv(index_csv).reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.images_dir, f"{row['image_id']}.jpg")).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, idx

ds = BRSETAllSplitsDataset(INDEX_CSV, IMAGES_DIR, transform=feat_tf)
print(f"Total images to encode: {len(ds):,}")

## Cell 4 — Extract DINOv2 CLS features for all 16,266 BRSET images

Cached to disk. ~15-20 min on T4.

In [ ]:
if FEATURES_PATH.exists():
    features = np.load(FEATURES_PATH)
    print(f"Loaded cached features: {features.shape}")
else:
    BS = 64
    loader = DataLoader(ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
    features = np.zeros((len(ds), EMBED_DIM), dtype=np.float32)
    print(f"Extracting features for {len(ds):,} images...")
    t0 = time.time()
    n_done = 0
    with torch.no_grad():
        for x, idxs in loader:
            x = x.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                cls = backbone(x)  # DINOv2 returns CLS feature directly: [B, 1024]
            features[idxs.numpy()] = cls.float().cpu().numpy()
            n_done += x.size(0)
            if n_done % 1024 == 0:
                rate = n_done / (time.time() - t0)
                eta = (len(ds) - n_done) / rate
                print(f"  {n_done:5d}/{len(ds):5d}  ({rate:.0f} img/s, ETA {eta/60:.1f} min)")
    np.save(FEATURES_PATH, features)
    print(f"Total: {(time.time()-t0)/60:.1f} min. Saved to {FEATURES_PATH}")

print(f"Features shape: {features.shape}  dtype: {features.dtype}")
print(f"Per-feature stats: mean={features.mean():.3f}  std={features.std():.3f}  "
      f"range=[{features.min():.2f}, {features.max():.2f}]")

## Cell 5 — Split features (reuse same patient-disjoint splits)

In [ ]:
df = pd.read_csv(INDEX_CSV).reset_index(drop=True)
labels = df[DISEASE_COLS].astype(np.float32).to_numpy()

train_mask = (df["split"] == "train").to_numpy()
val_mask   = (df["split"] == "val").to_numpy()
test_mask  = (df["split"] == "test").to_numpy()

X_train, y_train = features[train_mask], labels[train_mask]
X_val,   y_val   = features[val_mask],   labels[val_mask]
X_test,  y_test  = features[test_mask],  labels[test_mask]
print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

test_df = df[test_mask].reset_index(drop=True)
test_sex = test_df["patient_sex"].astype(int).to_numpy()
test_cam = test_df["camera"].astype(str).to_numpy()
test_qual = test_df["quality"].astype(str).to_numpy()
test_age = test_df["patient_age"].apply(lambda x: float(x) if pd.notna(x) else -1.0).to_numpy()

## Cell 6 — Train linear probe (same recipe as RETFound)

In [ ]:
n_pos = y_train.sum(axis=0)
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor(n_neg / np.maximum(n_pos, 1), dtype=torch.float32).to(DEVICE)

probe = nn.Linear(EMBED_DIM, N_LABELS).to(DEVICE)
optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32).to(DEVICE)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(DEVICE)

EPOCHS = 100
BS = 256
PATIENCE = 10
best_val_auroc = -1.0; best_state = None; no_improve = 0
n_train = len(X_train_t)

for epoch in range(EPOCHS):
    probe.train()
    perm = torch.randperm(n_train)
    losses = []
    for i in range(0, n_train, BS):
        idx = perm[i:i+BS]
        logits = probe(X_train_t[idx])
        loss = criterion(logits, y_train_t[idx])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    train_loss = float(np.mean(losses))

    probe.eval()
    with torch.no_grad():
        val_logits = probe(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
    val_y = y_val_t.cpu().numpy()
    aurocs = []
    for i in range(N_LABELS):
        if 0 < val_y[:, i].sum() < len(val_y):
            aurocs.append(roc_auc_score(val_y[:, i], val_probs[:, i]))
    val_macro = float(np.mean(aurocs))
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  val_macro_auroc={val_macro:.4f}")
    if val_macro > best_val_auroc:
        best_val_auroc = val_macro
        best_state = {k: v.clone() for k, v in probe.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}"); break

probe.load_state_dict(best_state)
print(f"\nBest val_macro_auroc = {best_val_auroc:.4f}")

## Cell 7 — Test eval: per-label AUROC + bootstrap CIs

In [ ]:
probe.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    test_logits = probe(X_test_t)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()

def bootstrap_auroc(p, y, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    aurocs, auprcs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if 0 < yi.sum() < len(yi):
            aurocs.append(roc_auc_score(yi, pi))
            auprcs.append(average_precision_score(yi, pi))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return dict(
        n=int(n), n_pos=int(y.sum()),
        auroc=float(np.mean(aurocs)) if aurocs else None,
        auroc_ci95=ci(aurocs) if aurocs else None,
        auprc=float(np.mean(auprcs)) if auprcs else None,
    )

per_label_results = {}
aurocs_overall = []
print("\n=== Per-label AUROC (DINOv2 linear probe on BRSET) ===")
for i, d in enumerate(DISEASE_COLS):
    r = bootstrap_auroc(test_probs[:, i], y_test[:, i])
    per_label_results[d] = r
    print(f"  {d:30s}  n_pos={r['n_pos']:4d}  AUROC={r['auroc']:.4f}  CI={r['auroc_ci95']}  AUPRC={r['auprc']:.4f}")
    aurocs_overall.append(r["auroc"])

macro_auroc = float(np.mean(aurocs_overall))
print(f"\nMacro AUROC: {macro_auroc:.4f}")

## Cell 8 — Fairness audit + significance test

In [ ]:
MIN_N = 30; MIN_POS = 10

def audit_subgroup(probs, y_true, mask, name):
    if mask.sum() < MIN_N: return None
    p, y = probs[mask], y_true[mask]
    out = {"subgroup": name, "n": int(mask.sum()), "per_label": {}}
    for i, d in enumerate(DISEASE_COLS):
        if y[:, i].sum() >= MIN_POS and y[:, i].sum() < len(y):
            out["per_label"][d] = bootstrap_auroc(p[:, i], y[:, i])
        else:
            out["per_label"][d] = None
    return out

audit = {}
audit["by_sex"] = [a for a in [
    audit_subgroup(test_probs, y_test, test_sex == 1, "sex=male"),
    audit_subgroup(test_probs, y_test, test_sex == 2, "sex=female"),
] if a]
audit["by_camera"] = [a for a in [
    audit_subgroup(test_probs, y_test, test_cam == c, f"camera={c}")
    for c in sorted(set(test_cam)) if c
] if a]
audit["by_quality"] = [a for a in [
    audit_subgroup(test_probs, y_test, test_qual == q, f"quality={q}")
    for q in sorted(set(test_qual)) if q
] if a]
known_age = test_age >= 0
audit["by_age_band"] = [a for a in [
    audit_subgroup(test_probs, y_test, known_age & (test_age >= lo) & (test_age < hi), f"age={name}")
    for (lo, hi, name) in [(0,40,"<40"),(40,60,"40-60"),(60,80,"60-80"),(80,200,"80+")]
] if a]

def find_gaps(rows, axis):
    print(f"\n=== Subgroup gaps where 95% CIs don't overlap ({axis}) ===")
    found = False
    for d in DISEASE_COLS:
        results = []
        for r in rows:
            v = r["per_label"].get(d)
            if v: results.append((r["subgroup"], v["auroc"], v["auroc_ci95"]))
        if len(results) < 2: continue
        for i in range(len(results)):
            for j in range(i+1, len(results)):
                a, b = results[i], results[j]
                if a[2][1] < b[2][0] or b[2][1] < a[2][0]:
                    found = True
                    print(f"  {d:30s}  {a[0]}: {a[1]:.3f} {a[2]}  vs  {b[0]}: {b[1]:.3f} {b[2]}")
    if not found: print("  (no statistically significant gaps)")

find_gaps(audit["by_sex"], "sex")
find_gaps(audit["by_camera"], "camera")
find_gaps(audit["by_quality"], "quality")
find_gaps(audit["by_age_band"], "age band")

## Cell 9 — Calibration (per-label ECE)

In [ ]:
def expected_calibration_error(probs, y_true, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0; total = len(y_true)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        mask = (probs >= lo) & (probs < hi)
        if i == n_bins - 1: mask = (probs >= lo) & (probs <= hi)
        if mask.sum() == 0: continue
        bin_acc = float(y_true[mask].mean())
        bin_conf = float(probs[mask].mean())
        ece += (mask.sum() / total) * abs(bin_acc - bin_conf)
    return float(ece)

per_label_ece = {}
print("\n=== Per-label ECE ===")
for i, d in enumerate(DISEASE_COLS):
    e = expected_calibration_error(test_probs[:, i], y_test[:, i])
    per_label_ece[d] = e
    print(f"  {d:30s}  ECE={e:.4f}")
macro_ece = float(np.mean(list(per_label_ece.values())))
print(f"\nMacro ECE: {macro_ece:.4f}")

## Cell 10 — Persist + comparison table vs RETFound

In [ ]:
results = {
    "method": "frozen_dinov2_vitl14_linear_probe",
    "best_val_macro_auroc": float(best_val_auroc),
    "test_n": int(len(y_test)),
    "test_macro_auroc": macro_auroc,
    "per_label": per_label_results,
    "fairness_audit": audit,
    "per_label_ece": per_label_ece,
    "macro_ece": macro_ece,
    "labels": DISEASE_COLS,
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
torch.save(probe.state_dict(), RESULTS / "linear_probe.pt")

print("\n=== DINOv2 results saved ===")
print(f"Macro AUROC: {macro_auroc:.4f}")
print(f"Macro ECE:   {macro_ece:.4f}")
print()
print("(Compare to RETFound linear probe: macro AUROC 0.7842, macro ECE 0.3038)")
print("(Compare to ResNet-50 baseline:     macro AUROC 0.9211)")

import shutil
zip_path = WORK / "dinov2_probe_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"\nWrote {zip_path}")